### Install torch and Hugging Face libraries

In [7]:
!pip install transformers datasets torch sentencepiece sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 14.0 MB/s eta 0:00:00 0:00:01


In [8]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Load the XNLI dataset from Hugging Face

In [4]:
from datasets import load_dataset, concatenate_datasets

dataset_full = load_dataset("xnli", "all_languages")
dataset_full = concatenate_datasets([dataset_full["train"],
                                     dataset_full["validation"],
                                     dataset_full["test"]])
# Random sample 50k elements
subset_size = 50000
dataset_reduced = dataset_full.shuffle(seed=42).select(range(subset_size))

dataset_reduced.shape, dataset_reduced.column_names

README.md:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

train-00000-of-00004.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

train-00001-of-00004.parquet:   0%|          | 0.00/239M [00:00<?, ?B/s]

train-00002-of-00004.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

train-00003-of-00004.parquet:   0%|          | 0.00/239M [00:00<?, ?B/s]

...

validation-00000-of-00001.parquet:   0%|          | 0.00/3.39M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5010 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2490 [00:00<?, ? examples/s]

((50000, 3), ['premise', 'hypothesis', 'label'])

### Extract Spanish and English data

In [5]:
def extract_language(example, lang):
    return {
        "premise": example["premise"][lang],
        "hypothesis": example["hypothesis"]["translation"]
         [example["hypothesis"]["language"].index(lang)],
        "label": example["label"]
    }

In [6]:
dataset_en = dataset_reduced.map(lambda x:
                                      extract_language(x, "en"),
                                      remove_columns=dataset_reduced.column_names)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [7]:
dataset_es = dataset_reduced.map(lambda x:
                                      extract_language(x, "es"),
                                      remove_columns=dataset_reduced.column_names)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

### Check an instance

In [8]:
dataset_es[0], dataset_en[0]

({'premise': 'La región se divide en una mitad oriental , alta normandía , a lo largo del valle del Sena , similar en el escenario a la ile-de-France ; y la más fuerte baja normandía hacia el oeste , más similar a la Vecina Bretaña .',
  'hypothesis': 'La región tiene siete tipos diferentes de tierra .',
  'label': 2},
 {'premise': 'The region divides into an eastern half , Haute-Normandie , along the Seine Valley , similar in scenery to the Ile-de-France ; and the more rugged Basse-Normandie to the west , more akin to neighboring Brittany .',
  'hypothesis': 'The region has seven different types of land .',
  'label': 2})

### Save Spanish and English datasets for later use

In [16]:
dataset_es.save_to_disk("/kaggle/working/xnli_50k_es")
print("Dataset 1 saved!")
dataset_en.save_to_disk("/kaggle/working/xnli_50k_en")
print("Dataset 2 saved!")

Saving the dataset (0/1 shards):   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset 1 saved!


Saving the dataset (0/1 shards):   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset 2 saved!


### Set up the MarianMT model for Spanish to Italian translation

In [9]:
from transformers import MarianMTModel, MarianTokenizer

# Load Spanish to Italian model
es_to_it_model_name = "Helsinki-NLP/opus-mt-es-it"
es_to_it_tokenizer = MarianTokenizer.from_pretrained(es_to_it_model_name)
es_to_it_model = MarianMTModel.from_pretrained(es_to_it_model_name).to(device)

### Batch translate texts using MarianMT

In [10]:
from tqdm import tqdm

def translate_texts(texts, model, tokenizer, batch_size=64, num_beams=5, max_length=256):
    translated_texts = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating...", 
                  total=len(texts)//batch_size):      
        batch = texts[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, 
                           truncation=True, max_length=max_length).to(device)
        with torch.no_grad(), torch.autocast(device_type="cuda"):
            outputs = model.generate(**inputs, num_beams=num_beams, max_length=max_length, early_stopping=True)
        translated_batch = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translated_texts.extend(translated_batch)

    return translated_texts

### Translate "premise" column from Spanish to Italian

In [13]:
italian_premises_from_es = [translate_texts(dataset_es["premise"], es_to_it_model, es_to_it_tokenizer)]

Translating...: 782it [32:31,  2.50s/it]                         


### Save Italian premises as a Hugging Face dataset for later use

In [18]:
from datasets import Dataset
it_from_es_prem = Dataset.from_dict({"translated_premises": italian_premises_from_es})

# Save as Hugging Face dataset
it_from_es_prem.save_to_disk("/kaggle/working/translated_premises_dataset")
print("Dataset saved!")

Saving the dataset (0/1 shards):   0%|          | 0/1 [00:00<?, ? examples/s]

Dataset saved!


### Load the Spanish and English datasets

In [11]:
from datasets import load_from_disk

dataset_es = load_from_disk("/kaggle/input/xnli-50k/xnli_50k_es")
dataset_en = load_from_disk("/kaggle/input/xnli-50k/xnli_50k_en", )

In [12]:
dataset_es[0], dataset_en[0]

({'premise': 'La región se divide en una mitad oriental , alta normandía , a lo largo del valle del Sena , similar en el escenario a la ile-de-France ; y la más fuerte baja normandía hacia el oeste , más similar a la Vecina Bretaña .',
  'hypothesis': 'La región tiene siete tipos diferentes de tierra .',
  'label': 2},
 {'premise': 'The region divides into an eastern half , Haute-Normandie , along the Seine Valley , similar in scenery to the Ile-de-France ; and the more rugged Basse-Normandie to the west , more akin to neighboring Brittany .',
  'hypothesis': 'The region has seven different types of land .',
  'label': 2})

### Translate "hypothesis" column from Spanish to Italian

In [ ]:
italian_hypotheses_from_es = [translate_texts(dataset_es["hypothesis"], es_to_it_model, es_to_it_tokenizer)]

In [ ]:
from datasets import Dataset

it_from_es_hyp = Dataset.from_dict({"translated_hypotheses": italian_hypotheses_from_es})

# Save as Hugging Face dataset
it_from_es_hyp.save_to_disk("/kaggle/working/translated_hypotheses_dataset")
print("Dataset saved!")

In [21]:
italian_premises_from_es = load_from_disk("/kaggle/input/xnli-50k/translated_premises_dataset")

In [40]:
italian_premises = italian_premises_from_es["translated_premises"][0]

In [44]:
italian_hypotheses_from_es = it_from_es_hyp
italian_hypotheses = italian_hypotheses_from_es["translated_hypotheses"][0]

In [45]:
italian_premises[0], italian_hypotheses[0]

('La regione è divisa in una metà orientale , alta normandia , lungo la valle della Senna , simile sul palco a ile-de-France; e la più bassa normandia verso ovest ,',
 'La regione ha sette diversi tipi di terra .')

In [47]:
labels = dataset_es["label"]

### Join the three columns into a dataset

In [50]:
dataset_it = {"premise": italian_premises, "hypothesis": italian_hypotheses, "label": labels}
dataset_it = Dataset.from_dict(dataset_it)

In [51]:
dataset_it

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 50000
})

In [54]:
dataset_it[0], dataset_es[0], dataset_en[0]

({'premise': 'La regione è divisa in una metà orientale , alta normandia , lungo la valle della Senna , simile sul palco a ile-de-France; e la più bassa normandia verso ovest ,',
  'hypothesis': 'La regione ha sette diversi tipi di terra .',
  'label': 2},
 {'premise': 'La región se divide en una mitad oriental , alta normandía , a lo largo del valle del Sena , similar en el escenario a la ile-de-France ; y la más fuerte baja normandía hacia el oeste , más similar a la Vecina Bretaña .',
  'hypothesis': 'La región tiene siete tipos diferentes de tierra .',
  'label': 2},
 {'premise': 'The region divides into an eastern half , Haute-Normandie , along the Seine Valley , similar in scenery to the Ile-de-France ; and the more rugged Basse-Normandie to the west , more akin to neighboring Brittany .',
  'hypothesis': 'The region has seven different types of land .',
  'label': 2})

In [56]:
len(dataset_it) == len(dataset_en)

True

### Build a new merged dataset with English and Italian data

In [57]:
merged_data = [
    {
        "premise": {"it": it["premise"], "en": en["premise"]},
        "hypothesis": {"it": it["hypothesis"], "en": en["hypothesis"]},
        "label": it["label"],
    }
    for it, en in zip(dataset_it, dataset_en)
]

# Convert to Hugging Face Dataset
merged_dataset = Dataset.from_list(merged_data)

In [59]:
merged_dataset, merged_dataset[0]

(Dataset({
     features: ['premise', 'hypothesis', 'label'],
     num_rows: 50000
 }),
 {'premise': {'en': 'The region divides into an eastern half , Haute-Normandie , along the Seine Valley , similar in scenery to the Ile-de-France ; and the more rugged Basse-Normandie to the west , more akin to neighboring Brittany .',
   'it': 'La regione è divisa in una metà orientale , alta normandia , lungo la valle della Senna , simile sul palco a ile-de-France; e la più bassa normandia verso ovest ,'},
  'hypothesis': {'en': 'The region has seven different types of land .',
   'it': 'La regione ha sette diversi tipi di terra .'},
  'label': 2})

### Save the merged dataset

In [64]:
merged_dataset.save_to_disk("/kaggle/working/italian_english_dataset")
print("Dataset saved!")

Saving the dataset (0/1 shards):   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset saved!


### Update the xnli-50k dataset to include only the final dataset

In [1]:
!pip install datasets torch
import torch
from datasets import load_from_disk, Dataset
italian_english_dataset = load_from_disk("/kaggle/input/xnli-50k/italian_english_dataset")
italian_english_dataset = Dataset.from_dict(italian_english_dataset.to_dict())

italian_english_dataset.save_to_disk("/kaggle/working/italian_english_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/50000 [00:00<?, ? examples/s]